# Bergant–Simpson Adelaide Rig (Independent DVCM Verification)

Compares rthym-moc DVCM against **published laboratory data** (He et al. 2025, Fig. 4 / Table 1), not golden rthym-moc JSON.

Mirrors **`tests/test_dvcm_bergant_adelaide_experiment.py`** (scalar peaks) and **`tests/test_dvcm_bergant_adelaide_trace.py`** (digitized experimental trace, peak-window gauge check).

Details: [`docs/bergant_adelaide_verification.md`](../../docs/bergant_adelaide_verification.md)

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jlillywh/RTHYM-MOC/main?labpath=validation%2Fnotebooks%2Fbergant_adelaide_verification.ipynb)

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rthym_moc
from _notebook_setup import bootstrap_validation_notebook

REPO_ROOT, TESTS_DIR, _VALIDATION = bootstrap_validation_notebook()
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(rthym_moc, '__version__', 'unknown')}")

## 2. Scalar peak checks (moderate + severe)

In [ ]:
from bergant_adelaide_verification_utils import CASES, CASE_LABELS, run_and_evaluate

for case_id in sorted(CASES):
    _, ref, m = run_and_evaluate(case_id)
    label = CASE_LABELS[case_id]
    print(
        f"{label}: peak PASS={m.passed_peak} "
        f"sim={m.sim_peak_kpa:.0f} exp={m.exp_peak_kpa:.0f} kPa abs rel={m.peak_rel_err:.3f}"
    )

## 3. Digitized Fig. 4 trace overlay

In [ ]:
from pathlib import Path
from bergant_adelaide_verification_utils import (
    SEVERE_VALVE_TRACE_CSV,
    evaluate_bergant_valve_trace,
    load_reference,
    load_valve_trace_csv,
    run_bergant_case,
    valve_gauge_pressure_kpa,
    validate_valve_trace_csv,
)

if not SEVERE_VALVE_TRACE_CSV.is_file():
    raise FileNotFoundError(
        'Missing digitized trace; see docs/bergant_adelaide_verification.md'
    )
errs = validate_valve_trace_csv(SEVERE_VALVE_TRACE_CSV)
assert not errs, errs
ref = load_reference('severe_cavitation')
trace = load_valve_trace_csv(SEVERE_VALVE_TRACE_CSV)
results = run_bergant_case(ref)
trace_m = evaluate_bergant_valve_trace('severe_cavitation', results, ref, trace)
t_s, p_g = valve_gauge_pressure_kpa(results)
t_e = trace['t_s']
p_e = trace['p_gauge_kPa']
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_e, p_e, 'o', ms=2, color='C1', label='Digitized experiment (gauge)')
ax.plot(t_s, p_g, '-', lw=1.2, color='C0', label='rthym-moc DVCM (gauge)')
ax.set_xlabel('t (s)')
ax.set_ylabel('P gauge (kPa)')
ax.set_title('Bergant Adelaide severe — valve pressure')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print(
    f"Trace peak check: PASS={trace_m.passed_peak} "
    f"exp={trace_m.exp_peak_gauge_kpa:.0f} sim={trace_m.sim_peak_gauge_kpa:.0f} kPa "
    f"rel={trace_m.peak_rel_err:.3f} (limit {trace_m.peak_rel_limit})"
)
print(f"RMS (informational): {trace_m.rms_kpa:.1f} kPa")

## 4. Summary

In [ ]:
scalar_ok = all(run_and_evaluate(c)[2].passed for c in CASES)
trace_ok = trace_m.passed_peak
print('Scalars:', 'PASS' if scalar_ok else 'FAIL')
print('Trace peak window:', 'PASS' if trace_ok else 'FAIL')
print('Overall:', 'PASS' if scalar_ok and trace_ok else 'FAIL')